### acceso a openlibrary (no sirve)

In [ ]:
# ======================================================================================
# ACCESO A API DE OPENLIBRARY
# ======================================================================================

import requests
import json
import pandas as pd
from pathlib import Path
import re


# ======================================================================================
# LECTURA DE DATOS
# ======================================================================================

# --- Contribuidores (editor, traductor...)
def leer_brief(isbn):   

    # Llamada para contribuidores
    url = f"http://openlibrary.org/api/volumes/brief/isbn/{isbn}"

    response = requests.get(url)

    if response.status_code == 200:
        api1 = response.json()

        if api1 == []:
            # No encuentra el libro
            return {}
        
        contenido = api1["records"]

        # clave tipo /books/OL9130631M
        clave = list(contenido.keys())[0]
        
        # Edición exacta
        edicion = contenido[clave]["details"]["details"]["edition_name"]

        # Editores
        contribuidores = contenido[clave]["details"]["details"]["contributors"]
        editores = []
        if contribuidores != "":
            for contr in contribuidores:
                if contr['role'] == 'Editor':
                    editores.append(contr["name"])

        # Peso
        peso = contenido[clave]["details"]["details"]['weight']

        # Dimensiones
        dim = contenido[clave]["details"]["details"]['physical_dimensions']

        # Formato
        formato = contenido[clave]["details"]["details"]['physical_format']

        # Categorías
        total_cats = contenido[clave]["data"]["subjects"]
        categorias = []
        for cat_dict in total_cats:
            categorias.append(cat_dict["name"])
        categorias += contenido[clave]["details"]["details"]['subjects']
        # categorias.apply(lambda x: x.translate(str.maketrans({"-":"", "/": "", "&": "and"})).strip())
        

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return {'edicion': edicion, 'editores': editores, 'peso': peso, 'dim': dim, 'formato': formato, 'subcategorias': categorias}

# --- Ratings
def leer_ratings(isbn):

    # Llamada para ratings
    url = f"https://openlibrary.org/search.json?isbn={isbn}&fields=rating*"

    response = requests.get(url)

    if response.status_code == 200:
        api2 = response.json()
        ratings = api2["docs"]

    else: 
        raise Exception(f"Conexión denegada. Status {response.status_code}")
    
    return ratings[0] if ratings else {}

# --- Sinopsis (ver como conseguirla de otro sitio)


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo_api(ruta_catalogos="data/prueba"): # cambiarlo para que te permita elegir cuántos libros hay que 

    print("Iniciando búsqueda en API...")
    catalogos = [f for f in Path(ruta_catalogos).iterdir() if f.is_file()]

    for ruta_cat in catalogos:
        print(f"\nLeyendo {ruta_cat.name}...")

        with open(ruta_cat, "r", encoding="utf-8") as f:
            catalogo = json.load(f)

        if 'openl' in catalogo[0].keys():
            print("Catálogo completo.")
            continue 
        
        # key = input("Continuar [Y/N]?")    

        # if key=='N':
        #     continue

        print(f"Comenzando con {ruta_cat.name}. Total de libros a buscar: {len(catalogo)+1}...")
        for libro in range(len(catalogo)):
            print(f"[{libro+1}/{len(catalogo)+1}]")
            ean = catalogo[libro]['EAN']

            brief = leer_brief(ean)
            ratings = leer_ratings(ean)

            if isinstance(brief, dict) and isinstance(ratings, dict):
                catalogo[libro].update({
                    **brief,
                    **ratings,
                    'portada': f"https://covers.openlibrary.org/b/isbn/{ean}-L.jpg",
                    'openl': True
                })
            else:
                catalogo[libro]['openl'] = False
                
            print("Catálogo terminado.")

        with open(ruta_cat, "w", encoding="utf-8") as f:
                    json.dump(catalogo, f, ensure_ascii=False, indent=2)


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

control_flujo_api()


## Capa silver nueva versión

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from constants import TRADUCTOR_EDITOR, OTROS_CONTRIBUIDORES, ILUSTRACIONES, ESCOLARES, CATEGORIAS, COLUMNAS_FINALES, CATEGORIAS, SUBCATEGORIAS, ENCUADERNACION, SPI_A_ED

# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

def crear_df(ruta_catalogos="data/bronze/catalogos"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50,"\nCreando DataFrame con todos los libros\n","="*50)
    for archivo in path.iterdir():
        if archivo.is_file():
            print(f"Añadiendo {archivo.name}")
            editorial = pd.read_json(archivo.absolute())
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=['EAN'], keep='first', inplace=True)

    # Limpiado de nombre de columnas
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))

    print("DataFrame creado con éxito.")
    return df


# =============================================================================
# FUNCIONES AUXILIARES Y LIMPIEZA BÁSICA
# =============================================================================

# Búsqueda de la moda en una lista de strings
def moda(x):
    """
    Devuelve la moda de una serie.
    """

    x = x.dropna()

    if len(x) == 0:
        return np.nan

    return x.mode().iloc[0]


# Extrae el número (precio, medida...) de un string
def extraer_numero(x):

    _NUMERO = re.compile(r"(\d+[.,]?\d*)")

    if pd.isna(x):
        return np.nan

    m = _NUMERO.search(str(x))

    if m is None:
        return np.nan

    return float(m.group(1).replace(",", "."))


# Conversión de los elementos de una lista/columna en listas
def normalizar_lista(valor):
    """
    Convierte cualquier valor en una lista.

    NaN -> []
    str -> [str]
    list -> list limpia
    ndarray -> list
    """

    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, str):
        valor = valor.strip().capitalize()
        if valor == "":
            return []

        return [valor]

    if isinstance(valor, np.ndarray):
        valor = valor.tolist()

    if isinstance(valor, (list, tuple)):
        salida = []
        for x in valor:
            if pd.isna(x):
                continue

            x = str(x).strip().capitalize()

            if x:
                salida.append(x)

        return list(dict.fromkeys(salida))

    return [str(valor)]


def normalizar_columnas_lista(df, columnas_listas):

    df = df.copy()

    for col in columnas_listas:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_lista)

    return df


# Limpieza básica del DF (eliminar filas son datos obligatorios, duplicados, relleno de columnas nulas y mapeo)
def limpieza_basica(df,dict_editoriales=SPI_A_ED,dict_encuadernacion=ENCUADERNACION):

    # quitar duplicados por EAN
    df.drop_duplicates(subset="ean", inplace=True)

    # eliminar libros sin autor y sin categoría
    df = df.dropna(
        subset=["ean", "titulo", "autoria", "categorias"],
        how="all",
    )

    # fecha
    df["fecha_publicacion"] = pd.to_datetime(
        df["fecha_publicacion"],
        format="%d-%m-%Y",
        errors="coerce",
    )

    # sinopsis
    df["sinopsis"] = df["sinopsis"].fillna("Sin sinopsis")

    # ids editoriales
    if dict_editoriales is not None:
        df["id_editorial"] = df["editorial"].map(dict_editoriales)

    # encuadernación
    if dict_encuadernacion is not None:
        df["encuadernacion"] = df["encuadernacion"].map(dict_encuadernacion)

    return df

def normalizar_titulos(nombre:str):
    articulos = {"El", "La", "Los", "Las", "Un", "Una", "Unos", "Unas"}

    if "," in nombre:
        titulo, articulo = map(str.strip, nombre.rsplit(",", 1))
        if articulo in articulos:
            nombre = f"{articulo} {titulo}"

    nombre = nombre.strip().capitalize()

    return nombre 

# =============================================================================
# MERGE DE COLUMNAS Y FEATURES
# =============================================================================

# Unión de columnas para crear otra nueva
def merge_columnas(df, nombre, columnas):

    df = df.copy()

    for col in columnas:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]

    df[nombre] = df[columnas].sum(axis=1).apply(lambda x: list(dict.fromkeys(x)))

    return df

# Coversión de columnas numéricas que aparecen como str
def extraer_numeros(df):

    df = df.copy()

    for col in ["precio","peso","grueso","n_paginas"]:
        if col in df.columns:
            df[col] = df[col].apply(extraer_numero)

    # dimensiones (ej: 240 x 170 mm)
    medidas = df["dimensiones"].astype(str).str.extract(r"(\d+[.,]?\d*)\D+(\d+[.,]?\d*)")

    df["alto_mm"] = medidas[0].str.replace(",", ".", regex=False).astype(float)
    df["ancho_mm"] = medidas[1].str.replace(",", ".", regex=False).astype(float)

    return df

def crear_marcadores(df, ilustraciones, escolares):
    # Escolares
    escolar = (
        df[escolares]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_escolar'] = escolar 

    # Ilustrada
    ilustrada = (
        df[ilustraciones]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_ilustrada'] = ilustrada

    # Impresión bajo demanda
    ibd = (
        df["ibd"]
        .str.len()
        > 0
    )
    df['es_ibd'] = ibd

    return df


def crear_portada(df:pd.DataFrame):
    # URL imagen
    ean = df["ean"].astype(str)
    df["img"] = (
        "https://static.cegal.es/imagenes/marcadas/"
        + ean.str[:8]
        + "/"
        + ean
        + ".gif"
    )

    return df


def definir_aparato_critico(df, cols_ap):
    mask = df[cols_ap].notna().any(axis=1)

    df["aparato_critico"] = mask

    df["tipo_aparato_critico"] = df[cols_ap].apply(lambda fila: [col for col in cols_ap if pd.notna(fila[col])],axis=1)

    df.loc[~mask, "tipo_aparato_critico"] = np.nan

    return df

def rellenar_columnas(df):

    df = df.copy()

    # Medidas por editorial + colección
    for col in ["alto_mm", "ancho_mm", "precio", "n_paginas"]:
        mediana_col = df.groupby(["editorial", "coleccion"])[col].transform("median")
        mediana_enc = df.groupby(["editorial", "encuadernacion"])[col].transform("median")
        
        df[col] = df[col].fillna(mediana_col).fillna(mediana_enc)


    # Grosor (fórmula estándar)
    df["grueso"] = df["grueso"].fillna(df["n_paginas"] * 0.04)

    # Peso (fórmula estándar)
    peso_estimado = df['peso'].fillna(df["alto_mm"] * df["ancho_mm"] * df["n_paginas"] * 0.08 + 120)

    df["peso"] = df["peso"].fillna(peso_estimado)

    # Idioma original 
    df["autor_principal"] = (
        df["autoria"]
        .apply(
            lambda x:
                x[0]
                if len(x)
                else np.nan
        )
    )

    idioma = (
        df.groupby("autor_principal")[
            "idioma_original"
        ]
        .transform(moda)
    )

    df["idioma_original"] = df["idioma_original"].fillna(idioma)

    df.drop(columns="autor_principal", inplace=True)

    return df


# =============================================================================
# LIMPIEZA FINAL
# =============================================================================

def limpiar_columnas(df, cols_borrar):

    borrar = [c for c in cols_borrar if c in df.columns]

    return df.drop(columns=borrar)


# =============================================================================
# PIPELINE
# =============================================================================

def limpiar_df_completa(data, dict_editoriales=SPI_A_ED, dict_encuadernacion=ENCUADERNACION):

    df = data.copy()

    # Cambio de nombres de columnas
    columnas_lista = list(set(
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
        + ESCOLARES
        + CATEGORIAS
        + ["autoria"]
    ))
    df = normalizar_columnas_lista(df, columnas_lista)

    # Limpieza
    df = limpieza_basica(df, dict_editoriales, dict_encuadernacion)
    df['titulo'] = df['titulo'].apply(normalizar_titulos)

    # Merge colaboradores
    df = merge_columnas(df, "traductor_y_editor", TRADUCTOR_EDITOR)
    df = merge_columnas(df,"otros_contribuidores", OTROS_CONTRIBUIDORES)
    df = merge_columnas(df, "subcategorias", CATEGORIAS)

    # Feature engineering
    df = extraer_numeros(df)
    df = definir_aparato_critico(df, OTROS_CONTRIBUIDORES)
    df = crear_marcadores(df, ESCOLARES, ILUSTRACIONES)
    df = crear_portada(df)

    # Relleno
    df = rellenar_columnas(df)

    # Limpieza final
    borrar = (
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
    )
    df = limpiar_columnas(df, borrar)

    df = df[[c for c in COLUMNAS_FINALES if c in df.columns]]

    return df


def validar_catalogo(df):
    # EAN únicos
    if df['ean'].nunique().count() < len(df['ean']):
        print('Existen números EAN repetidos.') 
    
    # sin nulos en columnas obligatorial
    cols_nulos = ["ean", "titulo", "autoria", "categorias"]
    for col in cols_nulos:
        if df[col].isna().sum() > 0:
            print(f'Existen nulos en la columna {col}.') 

    # fechas válidas
    formato = '%d/%m/%Y'
    for fecha in df['fecha_publicacion']:
        try:
            fecha_valida = datetime.strptime(fecha, formato)
            print("Fecha correcta")
        except ValueError:
            print("Fecha inválida")

    # dimensiones positivas
    cols_numericas = ["n_paginas","precio","alto_mm","ancho_mm","grueso","peso"]
    for col in cols_numericas:
        if any(x<=0 for x in df[col]):
            print(f"La columna {col} tiene núemeros no positivos.")


In [ ]:
# faltan:
# inferencia categorías
def inferencia_categoria(df, categorias=SUBCATEGORIAS):
    pass 

# hacer diccionario con categorias y subcategorias